## Section 1: Environment Setup
IMPORTANT: Dataset metadata is downloaded and loaded using the functions defined in [`src/metadata_loader.py`](../src/metadata_loader.py).

In [36]:
import random
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 21
random.seed(SEED)
np.random.seed(SEED)

# Repo config
REPO_URL = "https://github.com/s26732-ctrl/ZUM_2026_KatarzynaWalkowicz.git"
REPO_DIR = Path("/content/ZUM_2026_KatarzynaWalkowicz")

# Dataset config
DATA_DIR = Path("/content/drive/MyDrive/Biome-Health-Project")
IMAGES_DIR = DATA_DIR / "images"
SPLITS_DIR = REPO_DIR / "splits"

print(f"Random seed: {SEED}")
print(f"Repository directory: {REPO_DIR}")
print(f"Dataset directory: {DATA_DIR}")

Random seed: 21
Repository directory: /content/ZUM_2026_KatarzynaWalkowicz
Dataset directory: /content/drive/MyDrive/Biome-Health-Project


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# Check if repo is in session, if not - clone
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )

sys.path.insert(0, str(REPO_DIR / "src"))
print("Repository ready")

Repository ready


In [4]:
from metadata_loader import download_metadata, load_metadata

metadata_path = download_metadata(DATA_DIR)
metadata = load_metadata(metadata_path)

camera_locations = {
    image["location"]
    for image in metadata["images"]
    if image.get("location") is not None
}

print(f"Metadata file: {metadata_path}")
print(f"Images: {len(metadata['images'])}")
print(f"Annotations: {len(metadata['annotations'])}")
print(f"Categories: {len(metadata['categories'])}")
print(f"Camera locations: {len(camera_locations)}")

Metadata already exists
Metadata file: /content/drive/MyDrive/Biome-Health-Project/biome.json
Images: 37075
Annotations: 39386
Categories: 100
Camera locations: 176




---




## 2. Dataset Overview

This section explores the dataset metadata to check the structure, potential data quality issues and class and camera-location distributions before preparing the experimental split

### 2.1 Metadata Structure


In [5]:
images_df = pd.DataFrame(metadata["images"])
annotations_df = pd.DataFrame(metadata["annotations"])
categories_df = pd.DataFrame(metadata["categories"])

print("Images:", images_df.shape)
print("Annotations:", annotations_df.shape)
print("Categories:", categories_df.shape)

Images: (37075, 5)
Annotations: (39386, 4)
Categories: (100, 2)


In [15]:
display(images_df.head())
display(annotations_df.head())
display(categories_df.head())

,file_name,id,location,width,height
0,2018_NB47_000508.JPG,2018_NB47_000508,NB47,2688,1512
1,2018_NB46_000062.JPG,2018_NB46_000062,NB46,2688,1512
2,2018_NB47_000527.JPG,2018_NB47_000527,NB47,2688,1512
3,2018_NB20_000097.JPG,2018_NB20_000097,NB20,2688,1512
4,2018_NB20_000100.JPG,2018_NB20_000100,NB20,2688,1512


,image_id,category_id,id,min_count
0,2018_NB47_000508,0,2018_NB47_000508_ann_000,1
1,2018_NB46_000062,1,2018_NB46_000062_ann_000,1
2,2018_NB47_000527,1,2018_NB47_000527_ann_000,1
3,2018_NB47_000527,2,2018_NB47_000527_ann_001,1
4,2018_NB20_000097,1,2018_NB20_000097_ann_000,12


,name,id
0,baboon,0
1,impala,1
2,hyena_spotted,2
3,gazelle_thomsons,3
4,hare,4


**Metadata structure summary:**
- 37075 images, 39386 annotations, 100 categories
- one image can have more than one annotation
- min_count means the minimal number of animals of that species visible on the image


### 2.2 Data Integrity

In [12]:
print("Missing values in images:")
print(images_df.isna().sum())

print("\nMissing values in annotations:")
print(annotations_df.isna().sum())

print("\nMissing values in categories:")
print(categories_df.isna().sum())

Missing values in images:
file_name    0
id           0
location     0
width        0
height       0
dtype: int64

Missing values in annotations:
image_id       0
category_id    0
id             0
min_count      0
dtype: int64

Missing values in categories:
name    0
id      0
dtype: int64


In [13]:
print("Duplicate image IDs:", images_df["id"].duplicated().sum())
print("Duplicate filenames:", images_df["file_name"].duplicated().sum())
print("Duplicate annotation IDs:", annotations_df["id"].duplicated().sum())
print("Duplicate category IDs:", categories_df["id"].duplicated().sum())

Duplicate image IDs: 0
Duplicate filenames: 0
Duplicate annotation IDs: 0
Duplicate category IDs: 0


In [16]:
invalid_image_refs = ~annotations_df["image_id"].isin(images_df["id"])
invalid_category_refs = ~annotations_df["category_id"].isin(categories_df["id"])
print("Annotations with unknown image IDs:", invalid_image_refs.sum())
print("Annotations with unknown category IDs:", invalid_category_refs.sum())

images_without_annotations = ~images_df["id"].isin(annotations_df["image_id"])
print("Images without annotations:", images_without_annotations.sum())

Annotations with unknown image IDs: 0
Annotations with unknown category IDs: 0
Images without annotations: 0


**Data integrity summary:**
- no missing values
- no duplicates
- no invalid references
- no images without annotations


### 2.3 Dataset Statistics

In [17]:
# Single-label vs multi-label
categories_per_image = (
    annotations_df.groupby("image_id")["category_id"]
    .nunique()
)

print("Categories per image:")
print(categories_per_image.value_counts().sort_index())

single_label_images = categories_per_image[categories_per_image == 1].index
multi_label_images = categories_per_image[categories_per_image > 1].index

print(f"\nSingle-label images: {len(single_label_images)}")
print(f"Multi-label images: {len(multi_label_images)}")

Categories per image:
category_id
1    34915
2     2015
3      139
4        6
Name: count, dtype: int64

Single-label images: 34915
Multi-label images: 2160


In [19]:
# Category distribution in single-label images
single_label_annotations = annotations_df[
    annotations_df["image_id"].isin(single_label_images)
]

category_counts = (
    single_label_annotations
    .groupby("category_id")["image_id"]
    .nunique()
    .reset_index(name="image_count")
    .merge(categories_df, left_on="category_id", right_on="id")
    .sort_values("image_count", ascending=False)
)

print(category_counts[["name", "image_count"]])

                name  image_count
9         wildebeest         5874
3   gazelle_thomsons         5712
8              zebra         3655
1             impala         3180
25             shoat         1487
..               ...          ...
79            hoopoe            1
89              ibis            1
94        kingfisher            1
91          hamerkop            1
93           sparrow            1

[99 rows x 2 columns]


In [20]:
# Camera location coverage
single_label_data = (
    images_df[images_df["id"].isin(single_label_images)]
    .merge(
        single_label_annotations[["image_id", "category_id"]].drop_duplicates(),
        left_on="id",
        right_on="image_id"
    )
)

location_counts = (
    single_label_data
    .groupby("category_id")["location"]
    .nunique()
    .reset_index(name="location_count")
)

category_stats = (
    category_counts
    .merge(location_counts, on="category_id")
    .sort_values("image_count", ascending=False)
)

print(category_stats[["name", "image_count", "location_count"]].head(30))

                     name  image_count  location_count
0              wildebeest         5874             143
1        gazelle_thomsons         5712             113
2                   zebra         3655             146
3                  impala         3180             136
4                   shoat         1487              47
5                 warthog         1221             134
6                    topi         1199              98
7                  cattle         1147              86
8           hyena_spotted         1028             138
9                 giraffe         1022             117
10                buffalo          791              50
11                  blank          630             176
12                   hare          592              88
13                  eland          579              64
14               elephant          486             101
15         gazelle_grants          484              69
16          vervet_monkey          449              32
17       h

**Dataset Statistics Summary**
- out of 37075 images, 34915 are single-label, while 2160 contain multiple labels and will be excluded
- 99 out of 100 categories are represented among single-label images. The distribution is very imbalanced, ranging from 1 to 5874 images per category
- category coverage varies a lot across camera locations. Some species are widely distributed in over 100 locations, while others appear in only a few locations


### 2.4 Preliminary Class Selection

Preliminary thresholds: 150 images and 20 camera locations. These are for exploration and do not determine the final class selection yet

In [21]:
MIN_IMAGES = 150
MIN_LOCATIONS = 20

class_candidates = category_stats[
    (category_stats["image_count"] >= MIN_IMAGES)
    & (category_stats["location_count"] >= MIN_LOCATIONS)
]

display(class_candidates[["name", "image_count", "location_count"]])
print(f"Candidate categories: {len(class_candidates)}")

,name,image_count,location_count
0,wildebeest,5874,143
1,gazelle_thomsons,5712,113
2,zebra,3655,146
3,impala,3180,136
4,shoat,1487,47
5,warthog,1221,134
6,topi,1199,98
7,cattle,1147,86
8,hyena_spotted,1028,138
9,giraffe,1022,117


Candidate categories: 27



**Mid class selection comment:**

Out of the 100 original categories, 27 meet the preliminary criteria of at least 150 single-label images and 20 camera locations. Domestic animals ('shoat', 'cattle') and non-species labels ('blank') are excluded.

Based on the findings, **12 wildlife species** have been pre-selected.
The final selection will be confirmed after verifying that all classes are adequately represented in the group-based train/validation/test split.

In [22]:
SELECTED_CLASSES = [
    "wildebeest",
    "gazelle_thomsons",
    "zebra",
    "impala",
    "warthog",
    "topi",
    "hyena_spotted",
    "giraffe",
    "buffalo",
    "elephant",
    "jackal",
    "baboon",
]

selected_stats = category_stats[
    category_stats["name"].isin(SELECTED_CLASSES)
]

display(selected_stats[["name", "image_count", "location_count"]])

print(f"Selected classes: {len(selected_stats)}")
print(f"Total images: {selected_stats['image_count'].sum()}")

,name,image_count,location_count
0,wildebeest,5874,143
1,gazelle_thomsons,5712,113
2,zebra,3655,146
3,impala,3180,136
5,warthog,1221,134
6,topi,1199,98
8,hyena_spotted,1028,138
9,giraffe,1022,117
10,buffalo,791,50
14,elephant,486,101


Selected classes: 12
Total images: 24789


### 2.5 Camera Location Analysis
Checking potential imbalances to see if the selected classes are suitable for a group-based train/validation/test split, where camera locations cannot overlap between subsets

In [23]:
selected_data = single_label_data[
    single_label_data["category_id"].isin(selected_stats["category_id"])
].copy()

selected_data = selected_data.merge(
    categories_df,
    left_on="category_id",
    right_on="id",
    suffixes=("", "_category")
)

print("Selected images:", len(selected_data))
print("Selected classes:", selected_data["name"].nunique())
print("Camera locations:", selected_data["location"].nunique())

Selected images: 24789
Selected classes: 12
Camera locations: 176


In [26]:
images_per_location = (
    selected_data.groupby("location")
    .size()
    .sort_values(ascending=False)
)

print("Images per camera location:")
print(images_per_location.describe())

print("\nTop 10 locations:")
print(images_per_location.head(10))

Images per camera location:
count    176.000000
mean     140.846591
std      128.398328
min        1.000000
25%       44.000000
50%      106.000000
75%      203.500000
max      694.000000
dtype: float64

Top 10 locations:
location
NB26     694
OMC03    618
MN33     595
MN11     572
OMC09    457
OMC22    456
MT03     436
MN45     429
NB01     421
OMC10    394
dtype: int64


In [27]:
species_location_counts = (
    selected_data.groupby(["name", "location"])
    .size()
    .reset_index(name="image_count")
)

species_concentration = (
    species_location_counts
    .groupby("name")["image_count"]
    .max()
    .reset_index(name="max_images_at_one_location")
    .merge(
        selected_stats[["name", "image_count", "location_count"]],
        on="name"
    )
)

species_concentration["max_location_share"] = (
    species_concentration["max_images_at_one_location"]
    / species_concentration["image_count"] * 100
)

display(
    species_concentration[
        ["name", "image_count", "location_count", "max_location_share"]
    ].sort_values("max_location_share", ascending=False)
)

,name,image_count,location_count,max_location_share
1,buffalo,791,50,17.572693
8,topi,1199,98,11.509591
0,baboon,294,72,9.523810
10,wildebeest,5874,143,9.295199
4,giraffe,1022,117,8.904110
9,warthog,1221,134,8.190008
2,elephant,486,101,6.172840
7,jackal,327,103,5.198777
3,gazelle_thomsons,5712,113,4.884454
6,impala,3180,136,4.811321


**Camera location analysis summary**

- the selected dataset contains 24789 images across 12 classes and 176 camera locations.
- image counts vary significantly between locations (1 - 694 images, median: 106), however all selected species are represented across at least 50 locations, and no species has more than 17.6% of its images concentrated in a single location
- we can conitnue with this selection, but the *final* class representation and image counts need to be verified separately for train/validation/test


---



## 3. Experimental Data Split

Target dataset split: training 70%, validation 15%, test 15% subsets
IMPORTANT:
- all images from one camera location need to be in the same subset, even if they contain different species
- all 12 classes need to be represented in every subset
- final set will remain untouched during model development


### 3.1 Group-based Split Prep

In [28]:
split_data = selected_data[
    ["image_id", "file_name", "location", "name"]
].rename(columns={"name": "label"}).copy()

print("Total images:", len(split_data))
print("Unique images:", split_data["image_id"].nunique())
print("Classes:", split_data["label"].nunique())
print("Camera locations:", split_data["location"].nunique())

display(split_data.head())

Total images: 24789
Unique images: 24789
Classes: 12
Camera locations: 176


,image_id,file_name,location,label
0,2018_NB47_000508,2018_NB47_000508.JPG,NB47,baboon
1,2018_NB46_000062,2018_NB46_000062.JPG,NB46,impala
2,2018_NB20_000097,2018_NB20_000097.JPG,NB20,impala
3,2018_NB20_000100,2018_NB20_000100.JPG,NB20,impala
4,2018_NB47_000530,2018_NB47_000530.JPG,NB47,hyena_spotted


In [31]:
assert len(split_data) == split_data["image_id"].nunique(), (
    "Duplicate image ids detected"
)

assert split_data.notna().all().all(), (
    "Missing values detected"
)

assert split_data["label"].nunique() == len(SELECTED_CLASSES), (
    "Wrong number of classes"
)

print("Split input validation passed")

Split input validation passed


### 3.2 Group-based train/validation/test split


In [32]:
from sklearn.model_selection import GroupShuffleSplit

# first split
first_split = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=SEED
)

train_idx, temp_idx = next(
    first_split.split(split_data, groups=split_data["location"])
)

train_df = split_data.iloc[train_idx].copy()
temp_df = split_data.iloc[temp_idx].copy()

# second split
second_split = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=SEED
)

val_idx, test_idx = next(
    second_split.split(temp_df, groups=temp_df["location"])
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Final test images:", len(test_df))

Train images: 16954
Validation images: 4088
Final test images: 3747


### 3.3 Split Validation


In [33]:
# Location separation
train_locations = set(train_df["location"])
val_locations = set(val_df["location"])
test_locations = set(test_df["location"])

assert train_locations.isdisjoint(val_locations)
assert train_locations.isdisjoint(test_locations)
assert val_locations.isdisjoint(test_locations)

assert (
    len(train_df) + len(val_df) + len(test_df)
    == len(split_data)
)

print("No camera-location overlap detected")
print("All images assigned to exactly one subset")

No camera-location overlap detected
All images assigned to exactly one subset


In [34]:
# split stats
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "images": [len(train_df), len(val_df), len(test_df)],
    "locations": [
        train_df["location"].nunique(),
        val_df["location"].nunique(),
        test_df["location"].nunique()
    ]
})

split_summary["image_pct"] = (
    split_summary["images"] / len(split_data) * 100
).round(2)

split_summary["location_pct"] = (
    split_summary["locations"]
    / split_data["location"].nunique() * 100
).round(2)

display(split_summary)

,split,images,locations,image_pct,location_pct
0,train,16954,123,68.39,69.89
1,validation,4088,26,16.49,14.77
2,test,3747,27,15.12,15.34


In [35]:
# class distribution
class_distribution = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts()
}).fillna(0).astype(int)

display(class_distribution)

print(
    "\nAll classes represented in every split:",
    (class_distribution > 0).all().all()
)

,train,validation,test
label,,,
baboon,192,52,50
buffalo,509,123,159
elephant,385,41,60
gazelle_thomsons,4033,1050,629
giraffe,603,204,215
hyena_spotted,605,190,233
impala,1826,672,682
jackal,222,53,52
topi,806,220,173



All classes represented in every split: True



**Split validation summary**

The group-based split successfully separates all 176 camera locations without overlap between subsets

- **Train:** 16954 images (68.39%), 123 locations
- **Validation:** 4088 images (16.49%), 26 locations
- **Final test:** 3747 images (15.12%), 27 locations

All 12 classes are represented in each subset. The image proportions are close to the intended 70/15/15 split, although class distributions vary between subsets.

**Conclusion:** The split meets the main experimental requirements and can be accepted.


### 3.4 Saving the Experimental Split


In [39]:
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

for name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    split_ids = df[["image_id", "location"]].rename(
        columns={"image_id": "id", "location": "group_id"}
    )

    split_ids.to_csv(SPLITS_DIR / f"{name}_ids.csv", index=False)

print("Split files saved")

Split files saved


In [40]:
# verification
for name, original_df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    saved_df = pd.read_csv(SPLITS_DIR / f"{name}_ids.csv")

    expected_df = original_df[["image_id", "location"]].rename(
        columns={"image_id": "id", "location": "group_id"}
    ).reset_index(drop=True)

    assert saved_df.equals(expected_df), f"Mismatch in {name} split"

    print(f"{name}: {len(saved_df)} images verified")

print("\nAll split files verified successfully")

train: 16954 images verified
validation: 4088 images verified
test: 3747 images verified

All split files verified successfully
